***
*Project:* [Artificial Neuroscience: metrology and engineering for Deep Learning using Linear Algebra](https://www.seresearch.qmul.ac.uk/cmai/people/msandler/#grants)

*Author:* Jingwei Liu, Postdoctoral Research Associate, School of EECS, Queen Mary University of London

[[Centre for Digital Music (C4DM)](https://www.c4dm.eecs.qmul.ac.uk/people/) & [Centre for Fundamentals of AI and Computational Theory](https://www.seresearch.qmul.ac.uk/cfcs/people/jiliu/)]
***

# <span style="background-color:darkorange; color:white; padding:2px 6px">Document 4_1</span> 

# Conv-Tas Net with 4s Training Files

*Updated:* Jan 27, 2026


In [1]:
"""
Conv-TasNet: A Neural Network for Audio Source Separation
=========================================================

This module implements Conv-TasNet, which separates mixed audio into individual sources.

Think of it like this:
- Input: A recording of 2 people talking at the same time
- Output: 2 separate recordings, one for each person

Architecture:
1. ENCODER: Converts audio waveform → learned representation
2. SEPARATOR (TCN): Learns to create "masks" that isolate each source
3. DECODER: Converts masked representation → separated audio waveforms
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from itertools import permutations
from typing import Optional, Tuple, List
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import time
from tqdm import tqdm
from IPython.display import Audio
from torch.utils.tensorboard import SummaryWriter
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import librosa
import soundfile as sf
torch.cuda.is_available()

True

In [2]:
# =============================================================================
# SECTION 1: NORMALIZATION LAYERS
# =============================================================================

class CumulativeLayerNorm(nn.Module):
    """
    Cumulative Layer Normalization for causal (real-time) processing.
    
    Why do we need this?
    --------------------
    In real-time audio processing, we can only use information from the PAST,
    not the future. This normalization computes statistics cumulatively,
    meaning at each time step, it only considers samples up to that point.
    
    Parameters
    ----------
    num_features : int -- Number of features/channels in the input.
    eps : float -- Small value to prevent division by zero. Default: 1e-8
    learnable : bool -- If True, learns scaling (gain) and shifting (bias) parameters.
    
    Input Shape: (Batch, Features, Time)
    Output Shape: (Batch, Features, Time) - same as input
    """
    
    def __init__(self, num_features: int, eps: float = 1e-8, learnable: bool = True):
        super().__init__()
        
        self.eps = eps
        self.num_features = num_features
        
        if learnable:
            # Learnable parameters to scale and shift the normalized output
            self.gain = nn.Parameter(torch.ones(1, num_features, 1))
            self.bias = nn.Parameter(torch.zeros(1, num_features, 1))
        else:
            # Fixed parameters (no learning)
            self.register_buffer('gain', torch.ones(1, num_features, 1))
            self.register_buffer('bias', torch.zeros(1, num_features, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply cumulative layer normalization.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (Batch, Features, Time)
        
        Returns
        -------
        torch.Tensor
            Normalized tensor of same shape
        """
        batch_size, num_channels, time_steps = x.size()
        
        # Step 1: Sum across channels for each time step
        step_sum = x.sum(dim=1)  # Shape: (Batch, Time)
        step_squared_sum = x.pow(2).sum(dim=1)  # Shape: (Batch, Time)
        
        # Step 2: Compute cumulative sums (running totals)
        cumulative_sum = torch.cumsum(step_sum, dim=1)  # Shape: (Batch, Time)
        cumulative_squared_sum = torch.cumsum(step_squared_sum, dim=1)
        
        # Step 3: Count how many values we've seen at each time step
        # At time t, we've seen (t+1) * num_channels values
        entry_count = torch.arange(
            num_channels, 
            num_channels * (time_steps + 1), 
            num_channels,
            dtype=x.dtype,
            device=x.device
        ).view(1, -1)  # Shape: (1, Time)
        
        # Step 4: Calculate cumulative mean and standard deviation
        cumulative_mean = cumulative_sum / entry_count
        cumulative_variance = (
            cumulative_squared_sum / entry_count 
            - cumulative_mean.pow(2)
        )
        cumulative_std = (cumulative_variance + self.eps).sqrt()
        
        # Step 5: Normalize (subtract mean, divide by std)
        cumulative_mean = cumulative_mean.unsqueeze(1)  # Add channel dimension
        cumulative_std = cumulative_std.unsqueeze(1)
        
        normalized = (x - cumulative_mean) / cumulative_std
        
        # Step 6: Apply learned scaling and bias
        return normalized * self.gain + self.bias

In [3]:
def padding_same(x, dilation, kernel_size, causal=False):
    """
    Calculate padding to maintain same size
    """
    # Calculate total padding needed
    effective_kernel_size = dilation * (kernel_size - 1) + 1
    total_padding = effective_kernel_size - 1
    
    if causal:
        # For causal convolution, all padding goes to the left (past)
        x_padded = F.pad(x, (total_padding, 0))
    else:
        # Split into left and right padding
        pad_left = total_padding // 2
        pad_right = total_padding - pad_left  # Handles odd padding
    
        # Apply asymmetric padding: F.pad format is (left, right)
        x_padded = F.pad(x, (pad_left, pad_right))
    
    return x_padded

In [4]:
# =============================================================================
# SECTION 2: TEMPORAL CONVOLUTIONAL NETWORK (TCN) COMPONENTS
# =============================================================================

class DepthwiseSeparableConv1d(nn.Module):
    """
    Depthwise Separable 1D Convolution Block with Skip Connections.

    Parameters
    ----------
    input_channels  : int  -- Number of input channels.
    hidden_channels : int  -- Number of channels in the hidden layer.
    kernel_size     : int  -- Size of the convolutional kernel.
    dilation        : int  -- Spacing between kernel elements.
    causal          : bool -- If True, only use past information.
    use_skip        : bool -- If True, output a skip connection in addition to residual.
    """

    def __init__(
        self,
        input_channels: int,
        hidden_channels: int,
        kernel_size: int,
        dilation: int = 1,
        causal: bool = False,
        use_skip: bool = True
    ):
        super().__init__()

        self.causal = causal
        self.use_skip = use_skip

        # ── Store for use inside forward() ──────────────────────────────────
        self.dilation = dilation
        self.kernel_size = kernel_size
        # ────────────────────────────────────────────────────────────────────

        # Layer 1: Pointwise (1×1) convolution to expand channels
        self.conv_expand = nn.Conv1d(input_channels, hidden_channels, kernel_size=1)

        # Layer 2: Depthwise convolution — padding is now handled by padding_same()
        self.conv_depthwise = nn.Conv1d(
            hidden_channels,
            hidden_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            groups=hidden_channels,  # depthwise
            padding=0               # ← was self.padding; now manual via padding_same
        )

        # Layer 3: Pointwise convolution to create residual output
        self.conv_residual = nn.Conv1d(hidden_channels, input_channels, kernel_size=1)

        # Optional skip connection output
        if use_skip:
            self.conv_skip = nn.Conv1d(hidden_channels, input_channels, kernel_size=1)

        # Activation functions
        self.activation1 = nn.PReLU()
        self.activation2 = nn.PReLU()

        # Normalisation layers
        if causal:
            self.norm1 = CumulativeLayerNorm(hidden_channels)
            self.norm2 = CumulativeLayerNorm(hidden_channels)
        else:
            self.norm1 = nn.GroupNorm(1, hidden_channels, eps=1e-8)
            self.norm2 = nn.GroupNorm(1, hidden_channels, eps=1e-8)

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Parameters
        ----------
        x : torch.Tensor -- (Batch, Channels, Time)

        Returns
        -------
        residual : torch.Tensor
        skip     : torch.Tensor | None
        """
        # Block 1: expand → activate → normalise
        out = self.norm1(self.activation1(self.conv_expand(x)))

        # Block 2: manual padding → depthwise conv → activate → normalise
        out = padding_same(out, self.dilation, self.kernel_size, self.causal)  # ← replaces built-in padding
        out = self.conv_depthwise(out)
        out = self.norm2(self.activation2(out))

        # Outputs
        residual = self.conv_residual(out)

        if self.use_skip:
            skip = self.conv_skip(out)
            return residual, skip
        else:
            return residual, None


In [5]:
class TemporalConvNet(nn.Module):
    """
    Temporal Convolutional Network (TCN) for sequence modeling.
    
    What is this?
    -------------
    The TCN is the "brain" of Conv-TasNet. It learns to create masks that separate different audio sources.
    
    Key feature: DILATED CONVOLUTIONS
    ---------------------------------
    Regular convolutions look at nearby samples. Dilated convolutions skip samples, allowing the network to "see" further back in time
    without using more parameters.
    
    Example with dilation [1, 2, 4]:
    - Layer 1 (dilation=1): looks at samples 0, 1, 2
    - Layer 2 (dilation=2): looks at samples 0, 2, 4
    - Layer 3 (dilation=4): looks at samples 0, 4, 8
    
    This exponentially increases the "receptive field" (how far back the network can see).
    
    Parameters
    ----------
    input_dim : int -- Number of input features.
    output_dim : int -- Number of output features.
    bottleneck_dim : int -- Reduced dimension for efficient processing.
    hidden_dim : int -- Hidden dimension in depthwise conv blocks.
    num_layers : int -- Number of conv layers per stack.
    num_stacks : int -- Number of times to repeat the stack (resets dilation each time).
    kernel_size : int -- Convolution kernel size.
    causal : bool -- If True, use causal convolutions for real-time processing.
    use_skip : bool -- If True, use skip connections between layers.
    """
    
    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        bottleneck_dim: int,
        hidden_dim: int,
        num_layers: int,
        num_stacks: int,
        kernel_size: int = 3,
        causal: bool = False,
        use_skip: bool = True
    ):
        super().__init__()
        
        self.use_skip = use_skip
        
        # Input normalization
        if causal:
            self.input_norm = CumulativeLayerNorm(input_dim)
        else:
            self.input_norm = nn.GroupNorm(1, input_dim, eps=1e-8)
        
        # Bottleneck layer: reduce dimensions for efficiency
        self.bottleneck = nn.Conv1d(input_dim, bottleneck_dim, kernel_size=1)
        
        # Build the stack of dilated convolution layers
        self.conv_layers = nn.ModuleList()
        self.receptive_field = 0
        
        for stack_idx in range(num_stacks):
            for layer_idx in range(num_layers):
                # Dilation doubles each layer: 1, 2, 4, 8, 16, ...
                dilation = 2 ** layer_idx
                
                self.conv_layers.append(
                    DepthwiseSeparableConv1d(
                        input_channels=bottleneck_dim,
                        hidden_channels=hidden_dim,
                        kernel_size=kernel_size,
                        dilation=dilation,
                        causal=causal,
                        use_skip=use_skip
                    )
                )
                
                # Track receptive field (how far back we can "see")
                if stack_idx == 0 and layer_idx == 0:
                    self.receptive_field += kernel_size
                else:
                    self.receptive_field += (kernel_size - 1) * dilation
        
        # Output layer
        self.output_layer = nn.Sequential(
            nn.PReLU(),
            nn.Conv1d(bottleneck_dim, output_dim, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Process encoded audio features to create separation masks.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (Batch, Features, Time)
        
        Returns
        -------
        torch.Tensor
            Output tensor (mask) of shape (Batch, Output_Features, Time)
        """
        # Normalize and reduce to bottleneck dimension
        out = self.bottleneck(self.input_norm(x))
        
        # Process through all conv layers
        if self.use_skip:
            skip_sum = 0.0
            for layer in self.conv_layers:
                residual, skip = layer(out)
                out = out + residual
                skip_sum = skip_sum + skip
            out = self.output_layer(skip_sum)
        else:
            for layer in self.conv_layers:
                residual, _ = layer(out)
                out = out + residual
            out = self.output_layer(out)
        
        return out

In [6]:
# =============================================================================
# SECTION 3: MAIN MODEL - Conv-TasNet
# =============================================================================

class ConvTasNet(nn.Module):
    """
    Conv-TasNet: Convolutional Time-domain Audio Separation Network.
    
    What does this do?
    ------------------
    Separates a mixed audio signal into individual source signals.
    Example: Input a recording of 2 people talking → Output 2 separate recordings
    
    How it works (3 stages):
    ------------------------
    1. ENCODER: 
       - Takes the raw audio waveform
       - Converts it to a learned representation (like a spectrogram, but learned)
       
    2. SEPARATOR (TCN):
       - Analyzes the encoded representation
       - Creates "masks" for each source (like highlighting one speaker)
       
    3. DECODER:
       - Takes the masked representations
       - Converts them back to audio waveforms
    
    Parameters
    ----------
    num_sources : int -- Number of sources to separate (e.g., 2 for two speakers). Default: 2
    encoder_dim : int -- Dimension of the encoder output. Default: 512
    feature_dim : int -- Dimension of TCN bottleneck features. Default: 128
    sample_rate : int -- Audio sample rate in Hz. Default: 8000
    window_ms : float -- Encoder window length in milliseconds. Default: 2
    num_layers : int --  Number of layers in each TCN stack. Default: 8
    num_stacks : int --  Number of TCN stacks. Default: 3
    kernel_size : int -- Kernel size for TCN convolutions. Default: 3
    causal : bool -- If True, use causal convolutions (for real-time). Default: False
    
    Example
    -------
    >>> model = ConvTasNet(num_sources=2)
    >>> mixed_audio = torch.randn(4, 16000)  # Batch of 4, 2 seconds at 8kHz
    >>> separated = model(mixed_audio)  # Shape: (4, 2, 16000)
    >>> speaker1 = separated[:, 0, :]  # First speaker
    >>> speaker2 = separated[:, 1, :]  # Second speaker
    """
    
    def __init__(
        self,
        num_sources: int = 2,
        encoder_dim: int = 512,
        feature_dim: int = 128,
        sample_rate: int = 8000,
        window_ms: float = 2,
        num_layers: int = 8,
        num_stacks: int = 3,
        kernel_size: int = 3,
        causal: bool = False,
        use_skip: bool = True
    ):
        super().__init__()
        
        # Store configuration
        self.num_sources = num_sources
        self.encoder_dim = encoder_dim
        
        # Calculate window size in samples
        # Example: 2ms at 8000Hz = 0.002 * 8000 = 16 samples
        self.window_size = int(sample_rate * window_ms / 1000)
        self.hop_size = self.window_size // 2  # 50% overlap
        
        # === ENCODER ===
        # Converts waveform to learned representation
        # Like computing a spectrogram, but with learned filters
        self.encoder = nn.Conv1d(
            in_channels=1,              # Mono audio
            out_channels=encoder_dim,   # Number of "frequency bins"
            kernel_size=self.window_size,
            stride=self.hop_size,       # Hop between windows
            bias=False
        )
        
        # === SEPARATOR (TCN) ===
        # Creates masks to separate sources
        self.separator = TemporalConvNet(
            input_dim=encoder_dim,
            output_dim=encoder_dim * num_sources,  # One mask per source
            bottleneck_dim=feature_dim,
            hidden_dim=feature_dim * 4,
            num_layers=num_layers,
            num_stacks=num_stacks,
            kernel_size=kernel_size,
            causal=causal,
            use_skip=use_skip
        )
        
        # Store receptive field for reference
        self.receptive_field = self.separator.receptive_field
        
        # === DECODER ===
        # Converts masked representation back to waveform
        self.decoder = nn.ConvTranspose1d(
            in_channels=encoder_dim,
            out_channels=1,
            kernel_size=self.window_size,
            stride=self.hop_size,
            bias=False
        )

    def _pad_signal(self, audio: torch.Tensor) -> Tuple[torch.Tensor, int]:
        """
        Pad the input audio to ensure proper encoding/decoding.
        
        Why padding?
        -----------
        The encoder window needs to align properly with the signal.
        We also add padding at the beginning/end to avoid edge effects.
        
        Parameters
        audio : torch.Tensor -- Input audio of shape (Batch, Time) or (Batch, 1, Time)
        
        Returns
        padded_audio : torch.Tensor -- Padded audio of shape (Batch, 1, PaddedTime)
        padding_amount : int -- Amount of padding added at the end (needed for unpadding later)
        """
        # Ensure 3D input: (Batch, Channels, Time)
        if audio.dim() == 2:
            audio = audio.unsqueeze(1)
        elif audio.dim() != 3:
            raise ValueError(f"Input must be 2D or 3D, got {audio.dim()}D")
        
        batch_size, _, num_samples = audio.size()
        
        # Calculate padding needed to align with window/hop
        remainder = (self.hop_size + num_samples % self.window_size) % self.window_size
        padding_end = self.window_size - remainder if remainder > 0 else 0
        
        # Pad the end if needed
        if padding_end > 0:
            audio = F.pad(audio, (0, padding_end))
        
        # Add padding at beginning and end for edge effects
        audio = F.pad(audio, (self.hop_size, self.hop_size))
        
        return audio, padding_end

    def forward(self, mixture: torch.Tensor) -> torch.Tensor:
        """
        Separate a mixed audio signal into individual sources.
        
        Parameters
        ----------
        mixture : torch.Tensor -- Mixed audio signal of shape (Batch, Time) or (Batch, 1, Time)
        
        Returns
        -------
        torch.Tensor -- Separated sources of shape (Batch, NumSources, Time)
            
        Example
        -------
        >>> model = ConvTasNet(num_sources=2)
        >>> mix = torch.randn(1, 16000)  # 2 seconds of audio
        >>> sources = model(mix)  # Shape: (1, 2, 16000)
        """
        # Step 1: Pad the input signal
        padded, padding_amount = self._pad_signal(mixture)
        batch_size = padded.size(0)
        
        # Step 2: ENCODE - Transform waveform to latent representation
        # Shape: (Batch, EncoderDim, TimeFrames)
        encoded = self.encoder(padded)
        
        # Step 3: SEPARATE - Generate masks using TCN
        # Shape: (Batch, EncoderDim * NumSources, TimeFrames)
        mask_output = self.separator(encoded)
        
        # Apply sigmoid to get masks between 0 and 1
        # Reshape to (Batch, NumSources, EncoderDim, TimeFrames)
        masks = torch.sigmoid(mask_output).view(
            batch_size, self.num_sources, self.encoder_dim, -1
        )
        
        # Step 4: Apply masks to encoded representation
        # Multiply encoded by each mask to isolate sources
        # encoded: (Batch, EncoderDim, Time) → unsqueeze → (Batch, 1, EncoderDim, Time)
        masked = encoded.unsqueeze(1) * masks  # (Batch, NumSources, EncoderDim, Time)
        
        # Step 5: DECODE - Transform back to waveforms
        # Process all sources at once by combining batch and source dimensions
        masked_flat = masked.view(batch_size * self.num_sources, self.encoder_dim, -1)
        decoded = self.decoder(masked_flat)  # (Batch*NumSources, 1, Time)
        
        # Step 6: Remove padding and reshape
        # Calculate where to trim
        start = self.hop_size
        end = -(padding_amount + self.hop_size) if padding_amount > 0 else -self.hop_size
        
        output = decoded[:, :, start:end].contiguous()
        output = output.view(batch_size, self.num_sources, -1)
        
        return output

In [7]:
# =============================================================================
# SECTION 4: LOSS FUNCTIONS
# =============================================================================
class SI_SNR_Loss(nn.Module):
    """
    Scale-Invariant Signal-to-Noise Ratio (SI-SNR) Loss.
    
    Overview
    --------
    SI-SNR measures separation quality while ignoring volume differences.
    It answers: "How much of the estimate is signal vs. noise?"
    
    Mathematical Formulation
    ------------------------
    Given estimated signal ŝ and target signal s:
    
        1. Optimal scaling:     α = <ŝ, s> / ||s||²
        2. Signal component:    s_target = α · s
        3. Noise component:     e_noise = ŝ - s_target
        4. SI-SNR (dB):         10 · log₁₀(||s_target||² / ||e_noise||²)
    
    Interpretation
    --------------
        SI-SNR (dB)  │  Quality
        ─────────────┼──────────────
           < 0       │  Very poor (more noise than signal)
           0 - 5     │  Poor
           5 - 10    │  Acceptable
          10 - 15    │  Good
          15 - 20    │  Very good
           > 20      │  Excellent
    
    Training Note
    -------------
    We return NEGATIVE SI-SNR so that minimizing loss = maximizing SI-SNR.
    """
    
    def __init__(self, zero_mean: bool = True, eps: float = 1e-8):
        super().__init__()
        self.zero_mean = zero_mean
        self.eps = eps
    
    def forward(
        self, 
        estimated: torch.Tensor, 
        target: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute SI-SNR loss (per-sample, not averaged).
        
        Parameters
        ----------
        estimated : torch.Tensor
            Estimated signals, shape (batch_size, num_samples)
        target : torch.Tensor
            Target signals, shape (batch_size, num_samples)
        
        Returns
        -------
        torch.Tensor
            Negative SI-SNR for each sample, shape (batch_size,)
            Lower values indicate better separation.
        """
        # ─────────────────────────────────────────────────────────────────────
        # Step 0: Remove DC offset (mean) for true scale invariance
        # ─────────────────────────────────────────────────────────────────────
        if self.zero_mean:
            estimated = estimated - estimated.mean(dim=-1, keepdim=True)
            target = target - target.mean(dim=-1, keepdim=True)
        
        # ─────────────────────────────────────────────────────────────────────
        # Step 1: Compute optimal scaling factor α
        # 
        #         <ŝ, s>        Σᵢ ŝᵢ · sᵢ
        #    α = ────────  =  ──────────────
        #         ||s||²         Σᵢ sᵢ²
        #
        # This projects ŝ onto s to find the best-fit scaling.
        # ─────────────────────────────────────────────────────────────────────
        dot_product = torch.sum(estimated * target, dim=-1, keepdim=True)
        target_energy = torch.sum(target ** 2, dim=-1, keepdim=True) + self.eps
        scaling_factor = dot_product / target_energy
        
        # ─────────────────────────────────────────────────────────────────────
        # Step 2: Decompose estimate into signal and noise
        #
        #    s_target = α · s           (signal component)
        #    e_noise  = ŝ - s_target    (everything else = noise)
        #
        #    ŝ ──────┬────────────► s_target (aligned with target)
        #            │
        #            └────────────► e_noise  (orthogonal residual)
        # ─────────────────────────────────────────────────────────────────────
        signal_component = scaling_factor * target
        noise_component = estimated - signal_component
        
        # ─────────────────────────────────────────────────────────────────────
        # Step 3: Compute SI-SNR in decibels
        #
        #                    ||s_target||²
        #    SI-SNR = 10 · log₁₀ ──────────────
        #                    ||e_noise||²
        # ─────────────────────────────────────────────────────────────────────
        signal_energy = torch.sum(signal_component ** 2, dim=-1) + self.eps
        noise_energy = torch.sum(noise_component ** 2, dim=-1) + self.eps
        
        si_snr_db = 10 * torch.log10(signal_energy / noise_energy)
        
        # Return negative for loss minimization (we want to MAXIMIZE SI-SNR)
        return -si_snr_db  # Shape: (batch_size,)


class PIT_SI_SNR_Loss(nn.Module):
    """
    Permutation Invariant Training (PIT) with SI-SNR Loss.
    
    The Problem
    -----------
    In source separation, model outputs have no inherent ordering:
    
        Model outputs:  [output_0, output_1]
        True sources:   [speaker_A, speaker_B]
        
        Which output corresponds to which speaker? We don't know!
    
    The Solution (PIT)
    ------------------
    Try ALL possible assignments and use the one with lowest loss:
    
        ┌─────────────────────────────────────────────────────────┐
        │  Permutation 1:  output_0 → speaker_A                   │
        │                  output_1 → speaker_B    →  loss = 2.3  │
        ├─────────────────────────────────────────────────────────┤
        │  Permutation 2:  output_0 → speaker_B                   │
        │                  output_1 → speaker_A    →  loss = 5.1  │
        └─────────────────────────────────────────────────────────┘
                                    ↓
                    Select Permutation 1 (lower loss)
    
    Per-Sample Selection
    --------------------
    IMPORTANT: Each sample in a batch gets its OWN best permutation!
    
        Sample 0: Best = perm (0,1)  ┐
        Sample 1: Best = perm (1,0)  │  Different samples,
        Sample 2: Best = perm (0,1)  │  different best perms!
        Sample 3: Best = perm (1,0)  ┘
    
    Complexity Note
    ---------------
    Number of permutations = n! where n = number of sources
    
        Sources │ Permutations
        ────────┼─────────────
           2    │      2
           3    │      6
           4    │     24
           5    │    120  (gets expensive!)
    
    Parameters
    ----------
    zero_mean : bool, default=True -- Use zero-mean SI-SNR computation.
    """
    
    def __init__(self, zero_mean: bool = True):
        super().__init__()
        self.si_snr_loss = SI_SNR_Loss(zero_mean=zero_mean)

    def forward(self, estimated: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        estimated : (Batch, NumSources, Samples)
        target    : (Batch, NumSources, Samples)

        Returns
        -------
        loss : scalar (batch-averaged best losses)
        """
        batch_size, num_sources, num_samples = estimated.size()
        perms = list(permutations(range(num_sources)))

        # Store per-sample loss for each permutation
        # Shape: (num_perms, batch_size)
        perm_losses = []

        for perm in perms:
            perm_estimated = estimated[:, perm, :]

            # Sum SI-SNR across sources for each sample
            # Shape: (batch_size,)
            perm_loss = torch.zeros(batch_size, device=estimated.device)
            for src_idx in range(num_sources):
                perm_loss = perm_loss + self.si_snr_loss(
                    perm_estimated[:, src_idx, :],
                    target[:, src_idx, :]
                )

            perm_losses.append(perm_loss / num_sources)

        # Stack: (num_perms, batch_size)
        perm_losses = torch.stack(perm_losses, dim=0)

        # Find best permutation for EACH sample independently
        # min_losses shape: (batch_size,)
        # best_perm_idx shape: (batch_size,)
        min_losses, best_perm_idx = perm_losses.min(dim=0)

        # Average across batch
        return min_losses.mean()

In [8]:
# =============================================================================
# SECTION 5: TRAINING SCRIPT
# =============================================================================

class Trainer:
    """
    Training manager for Conv-TasNet.
    
    Handles:
    - Training loop
    - Validation
    - Checkpointing
    - Logging to TensorBoard
    - Learning rate scheduling  # ← ADDED to docstring
    
    Parameters
    ----------
    model : nn.Module -- Conv-TasNet model
    train_loader : DataLoader -- Training data loader
    val_loader : DataLoader -- Validation data loader
    optimizer : torch.optim.Optimizer -- Optimizer (e.g., Adam)
    scheduler : torch.optim.lr_scheduler._LRScheduler -- Learning rate scheduler (optional)  # ← ADDED
    device : str -- Device to train on ('cuda' or 'cpu')
    checkpoint_dir : str -- Directory to save checkpoints
    log_dir : str -- Directory for TensorBoard logs
    """
    
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        optimizer: torch.optim.Optimizer,
        scheduler: Optional[torch.optim.lr_scheduler._LRScheduler] = None,  # ← ADDED parameter
        device: str = 'cuda',
        checkpoint_dir: str = './checkpoints',
        log_dir: str = './logs'
    ):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = optimizer
        self.scheduler = scheduler  # ← ADDED: Store the scheduler
        self.device = device
        
        # Loss function
        self.criterion = PIT_SI_SNR_Loss(zero_mean=True)
        
        # Checkpointing
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        
        # Logging
        self.writer = SummaryWriter(log_dir)
        
        # Training state
        self.epoch = 0
        self.global_step = 0
        self.best_val_loss = float('inf')
    
    def train_epoch(self) -> Tuple[float, float]:
        """
        Train for one epoch.
        
        Returns
        -------
        avg_loss : float -- Average training loss
        avg_si_snr : float -- Average SI-SNR in dB
        """
        self.model.train()
        total_loss = 0.0
        
        for batch_idx, (mixture, sources) in enumerate(self.train_loader):
            # Move to device
            mixture = mixture.to(self.device)  # (Batch, Samples)
            sources = sources.to(self.device)  # (Batch, NumSources, Samples)
            
            # Forward pass
            estimated = self.model(mixture)  # (Batch, NumSources, Samples)
            
            # Calculate loss
            loss = self.criterion(estimated, sources)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping (helps with stability)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)
            
            self.optimizer.step()
            
            # Accumulate metrics
            total_loss += loss.item()
            
            # Log to TensorBoard
            if batch_idx % 500 == 0:
                self.writer.add_scalar('Train/Loss', loss.item(), self.global_step)
            
            self.global_step += 1
            
            # Print progress
            if batch_idx % 1000 == 0:
                print(f"  Batch {batch_idx}/{len(self.train_loader)}, "
                      f"Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(self.train_loader)
        
        return avg_loss
    
    @torch.no_grad()
    def validate(self) -> Tuple[float, float]:
        """
        Validate the model.
        
        Returns
        -------
        avg_loss : float -- Average validation loss
        """
        self.model.eval()
        total_loss = 0.0
        
        for mixture, sources in self.val_loader:
            mixture = mixture.to(self.device)
            sources = sources.to(self.device)
            
            # Forward pass
            estimated = self.model(mixture)
            
            # Calculate loss
            loss = self.criterion(estimated, sources)
            total_loss += loss.item()
        
        avg_loss = total_loss / len(self.val_loader)
        
        return avg_loss
    
    def save_checkpoint(self, filename: str = 'checkpoint.pth'):
        """Save training checkpoint."""
        checkpoint = {
            'epoch': self.epoch,
            'global_step': self.global_step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
        }
        # ← ADDED: Save scheduler state if it exists
        if self.scheduler is not None:
            checkpoint['scheduler_state_dict'] = self.scheduler.state_dict()
        
        torch.save(checkpoint, self.checkpoint_dir / filename)
        print(f"  ✓ Checkpoint saved: {filename}")
    
    def load_checkpoint(self, filename: str = 'checkpoint.pth'):
        """Load training checkpoint."""
        checkpoint_path = self.checkpoint_dir / filename
        if not checkpoint_path.exists():
            print(f"  ✗ Checkpoint not found: {filename}")
            return False
        
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.epoch = checkpoint['epoch']
        self.global_step = checkpoint['global_step']
        self.best_val_loss = checkpoint['best_val_loss']
        
        # ← ADDED: Load scheduler state if it exists
        if self.scheduler is not None and 'scheduler_state_dict' in checkpoint:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        print(f"  ✓ Checkpoint loaded: {filename} (epoch {self.epoch})")
        return True
    
    def _get_current_lr(self) -> float:  # ← ADDED: Helper method to get current learning rate
        """Get the current learning rate from the optimizer."""
        return self.optimizer.param_groups[0]['lr']
    
    def find_latest_checkpoint(self) -> Optional[str]:
        """Find the most recent checkpoint file."""
        checkpoints = list(self.checkpoint_dir.glob('checkpoint_epoch_*.pth'))
        
        if not checkpoints:
            return None
        
        # Extract epoch numbers and find the latest
        def get_epoch_num(path):
            # Extract number from 'checkpoint_epoch_X.pth'
            name = path.stem  # 'checkpoint_epoch_X'
            return int(name.split('_')[-1])
        
        latest = max(checkpoints, key=get_epoch_num)
        return latest.name

    def train(self, num_epochs: int, resume_from: Optional[str] = None):
        """
        Main training loop.
        
        Parameters
        ----------
        num_epochs : int -- Total number of epochs to train
        resume_from : str, optional -- Checkpoint filename to resume from
        """
        # ─────────────────────────────────────────────────────────────────────
        # Handle checkpoint resumption
        # ─────────────────────────────────────────────────────────────────────
        start_epoch = 0
        
        if resume_from is not None:
            if self.load_checkpoint(resume_from):
                start_epoch = self.epoch + 1  # Start from NEXT epoch
                print(f"  ▶ Resuming from epoch {start_epoch}")
            else:
                print(f"  ⚠ Could not load checkpoint, starting from scratch")
        
        print("=" * 70)
        print("TRAINING CONV-TASNET")
        print("=" * 70)
        print(f"Device: {self.device}")
        print(f"Total Epochs: {num_epochs}")
        print(f"Starting Epoch: {start_epoch + 1}")  # Human-readable (1-indexed)
        print(f"Training samples: {len(self.train_loader.dataset)}")
        print(f"Validation samples: {len(self.val_loader.dataset)}")
        print(f"Current LR: {self._get_current_lr():.6f}")
        print(f"Best Val Loss: {self.best_val_loss:.4f}")
        print(f"Scheduler: {type(self.scheduler).__name__ if self.scheduler else 'None'}")
        print("=" * 70)
        
        # ─────────────────────────────────────────────────────────────────────
        # Training loop - START FROM start_epoch, NOT 0
        # ─────────────────────────────────────────────────────────────────────
        if start_epoch == 0:
            self.save_checkpoint(f'checkpoint_epoch_{start_epoch}.pth')  # Save initial checkpoint

        for epoch in range(start_epoch, num_epochs):  # ← Changed!
            self.epoch = epoch
            epoch_start_time = time.time()
            
            current_lr = self._get_current_lr()
            print(f"\nEpoch {epoch + 1}/{num_epochs} (LR: {current_lr:.6f})")
            print("-" * 70)
            
            # Train
            train_loss = self.train_epoch()
            
            # Validate
            val_loss = self.validate()
            
            epoch_time = time.time() - epoch_start_time
            
            # Step scheduler
            if self.scheduler is not None:
                if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss)
                else:
                    self.scheduler.step()
            
            # Log to TensorBoard (uses epoch as x-axis, so logs continue correctly)
            self.writer.add_scalar('Epoch/Train_Loss', train_loss, epoch)
            self.writer.add_scalar('Epoch/Val_Loss', val_loss, epoch)
            self.writer.add_scalar('Epoch/Learning_Rate', self._get_current_lr(), epoch)
            
            # Print summary
            print(f"\nEpoch {epoch + 1} Summary:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss:   {val_loss:.4f}")
            print(f"  LR:         {self._get_current_lr():.6f}")
            print(f"  Time:       {epoch_time:.2f}s")
            
            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.save_checkpoint('best_model.pth')
                print(f"  🌟 New best model!")

            # Save checkpoint
            self.save_checkpoint(f'checkpoint_epoch_{epoch + 1}.pth')
        
        print("\n" + "=" * 70)
        print("TRAINING COMPLETE!")
        print(f"Final LR: {self._get_current_lr():.6f}")
        print("=" * 70)
        self.writer.close()

In [10]:
config = {
    # Model parameters
    'num_sources': 2,
    'encoder_dim': 512,
    'feature_dim': 128,
    'sample_rate': 8000,
    'window_ms': 2,
    'num_layers': 8,
    'num_stacks': 3,
    'kernel_size': 3,
    'causal': False,
    'use_skip': True,
    
    # Training parameters
    'batch_size': 6,
    'num_epochs': 100,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    
    # System parameters
    'device': 'cuda',
    'num_workers': 4,
    'checkpoint_dir': './checkpoints',
    'log_dir': './logs',
}

model = ConvTasNet(
    num_sources=config['num_sources'],
    encoder_dim=config['encoder_dim'],
    feature_dim=config['feature_dim'],
    sample_rate=config['sample_rate'],
    window_ms=config['window_ms'],
    num_layers=config['num_layers'],
    num_stacks=config['num_stacks'],
    kernel_size=config['kernel_size'],
    causal=config['causal'],
    use_skip=config['use_skip']
)

## Lazy Loading Audio Dataset with Fixed-Length Segments

In [10]:
class PairedAudioDataset(Dataset):
    """
    Same as PairedAudioDataset but with safety checks for edge cases.
    Use this if you're not 100% sure all files are exactly 4 seconds.
    """
    
    def __init__(
        self,
        mixed_dir: str,
        source_dirs: list,
        sample_rate: int = 8000,
        expected_duration: float = 4.0
    ):
        self.mixed_dir = mixed_dir
        self.source_dirs = source_dirs
        self.sample_rate = sample_rate
        self.expected_length = int(sample_rate * expected_duration)
        
        # Get files that exist in ALL directories
        mixed_files = set(f for f in os.listdir(mixed_dir) if f.endswith('.wav'))
        
        for src_dir in source_dirs:
            src_files = set(f for f in os.listdir(src_dir) if f.endswith('.wav'))
            mixed_files = mixed_files.intersection(src_files)
        
        self.file_list = sorted(list(mixed_files))
        print(f"📁 Found {len(self.file_list)} matching audio files")
    
    def __len__(self):
        return len(self.file_list)
    
    def _ensure_length(self, audio: np.ndarray) -> np.ndarray:
        """Ensure audio is exactly the expected length."""
        if len(audio) > self.expected_length:
            # Trim if too long
            return audio[:self.expected_length]
        elif len(audio) < self.expected_length:
            # Pad if too short
            pad_len = self.expected_length - len(audio)
            return np.pad(audio, (0, pad_len), mode='constant')
        return audio
    
    def __getitem__(self, idx):
        filename = self.file_list[idx]
        
        # Load mixed audio
        mixed_path = os.path.join(self.mixed_dir, filename)
        mixed_audio, _ = sf.read(mixed_path)
        mixed_audio = self._ensure_length(mixed_audio.mean(axis=1))  # Convert to mono if stereo
        
        # Load source audio files
        source_audios = []
        for src_dir in self.source_dirs:
            src_path = os.path.join(src_dir, filename)
            src_audio, _ = sf.read(src_path)
            src_audio = self._ensure_length(src_audio.mean(axis=1))  # Convert to mono if stereo
            source_audios.append(src_audio)
        
        # Convert to tensors
        mixed_tensor = torch.tensor(mixed_audio, dtype=torch.float32).unsqueeze(0)
        sources_tensor = torch.tensor(np.stack(source_audios), dtype=torch.float32)
        
        return mixed_tensor, sources_tensor

In [10]:
# For validation
# Load mixed and source audio
mixed_dir_val = 'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix\dev-clean\mix_clean'
source_dirs_val = ['D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix\dev-clean\s1', \
                   'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix\dev-clean\s2']  # List of source directories

In [20]:
def main():

    val_dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_val,
        source_dirs=source_dirs_val,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    for mixed, sources in val_loader:
        print(f"Mixed shape: {mixed.shape}")  # Should be (Batch, 1, Samples)
        print(f"Sources shape: {sources.shape}")  # Should be (Batch, NumSources, Samples)
        break  # Just check one batch

if __name__ == "__main__":
    main()

📁 Found 3000 matching audio files
Mixed shape: torch.Size([6, 1, 32000])
Sources shape: torch.Size([6, 2, 32000])


In [ ]:
%load_ext tensorboard
%tensorboard --logdir config['log_dir']

In [ ]:
def main(resume: bool = False):
    config = {
        # Model parameters
        'num_sources': 2,
        'encoder_dim': 512,
        'feature_dim': 128,
        'sample_rate': 8000,
        'window_ms': 2,
        'num_layers': 8,
        'num_stacks': 3,
        'kernel_size': 3,
        'causal': False,
        'use_skip': True,
        
        # Training parameters
        'batch_size': 6,
        'num_epochs': 100,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        
        # System parameters
        'device': 'cuda',
        'num_workers': 0,
        'checkpoint_dir': './checkpoints',
        'log_dir': './logs',
    }

    # Initialize model
    model = ConvTasNet(
        num_sources=config['num_sources'],
        encoder_dim=config['encoder_dim'],
        feature_dim=config['feature_dim'],
        sample_rate=config['sample_rate'],
        window_ms=config['window_ms'],
        num_layers=config['num_layers'],
        num_stacks=config['num_stacks'],
        kernel_size=config['kernel_size'],
        causal=config['causal'],
        use_skip=config['use_skip']
    ).to(config['device'])
    
    # Initialize optimizer and scheduler
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config['learning_rate'], 
        weight_decay=config['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=3
    )
    
    # For training
    # Load mixed and source audio
    mixed_dir_train = "D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\\train-clean-100_4s\mix_clean"
    source_dirs_train = ["D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\\train-clean-100_4s\s1", \
                        "D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\\train-clean-100_4s\s2"]  # List of source directories
    train__dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_train,
        source_dirs=source_dirs_train,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    
    train_loader = DataLoader(
            train__dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=config['num_workers'],
            pin_memory=True
        )
    
    # For validation
    # Load mixed and source audio
    mixed_dir_val = 'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\dev_clean_4s\mix_clean'
    source_dirs_val = ['D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\dev_clean_4s\s1', \
                    'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix_4s\dev_clean_4s\s2']  # List of source directories
    val_dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_val,
        source_dirs=source_dirs_val,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,  # Validation should not shuffle
            num_workers=config['num_workers'],
            pin_memory=True
        )

    # Initialize trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,  # ← ADDED: Pass the scheduler to the trainer
        device=config['device'],
        checkpoint_dir=config['checkpoint_dir'],
        log_dir=config['log_dir']
    )
    
    # Check for existing checkpoint if resuming
    resume_checkpoint = None
    if resume:
        resume_checkpoint = trainer.find_latest_checkpoint()
        if resume_checkpoint:
            print(f"Found checkpoint: {resume_checkpoint}")
        else:
            print("No checkpoint found, starting fresh")
    
    # Start training
    trainer.train(
        num_epochs=config['num_epochs'],
        resume_from=resume_checkpoint  # ← Pass checkpoint name
    )


if __name__ == "__main__":
    main(resume=False)


📁 Found 33640 matching audio files
📁 Found 3812 matching audio files
TRAINING CONV-TASNET
Device: cuda
Total Epochs: 100
Starting Epoch: 1
Training samples: 33640
Validation samples: 3812
Current LR: 0.001000
Best Val Loss: inf
Scheduler: ReduceLROnPlateau
  ✓ Checkpoint saved: checkpoint_epoch_0.pth

Epoch 1/100 (LR: 0.001000)
----------------------------------------------------------------------
  Batch 0/5607, Loss: 19.9723


KeyboardInterrupt: 

In [13]:
def main(resume: bool = False):
    config = {
        # Model parameters
        'num_sources': 3,
        'encoder_dim': 512,
        'feature_dim': 128,
        'sample_rate': 8000,
        'window_ms': 2,
        'num_layers': 8,
        'num_stacks': 3,
        'kernel_size': 3,
        'causal': False,
        'use_skip': True,
        
        # Training parameters
        'batch_size': 6,
        'num_epochs': 100,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        
        # System parameters
        'device': 'cuda',
        'num_workers': 0,
        'checkpoint_dir': './checkpoints',
        'log_dir': './logs',
    }

    # Initialize model
    model = ConvTasNet(
        num_sources=config['num_sources'],
        encoder_dim=config['encoder_dim'],
        feature_dim=config['feature_dim'],
        sample_rate=config['sample_rate'],
        window_ms=config['window_ms'],
        num_layers=config['num_layers'],
        num_stacks=config['num_stacks'],
        kernel_size=config['kernel_size'],
        causal=config['causal'],
        use_skip=config['use_skip']
    ).to(config['device'])
    
    # Initialize optimizer and scheduler
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config['learning_rate'], 
        weight_decay=config['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=3
    )
    
    # For training
    # Load mixed and source audio
    mixed_dir_train = "D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\\train-clean-100_4s\mix_clean"
    source_dirs_train = ["D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\\train-clean-100_4s\s1", \
                        "D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\\train-clean-100_4s\s2", \
                        "D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\\train-clean-100_4s\s3"]  # List of source directories
    train__dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_train,
        source_dirs=source_dirs_train,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    
    train_loader = DataLoader(
            train__dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=config['num_workers'],
            pin_memory=True
        )
    
    # For validation
    # Load mixed and source audio
    mixed_dir_val = 'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\dev-clean_4s\mix_clean'
    source_dirs_val = ['D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\dev-clean_4s\s1', \
                    'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\dev-clean_4s\s2', \
                    'D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri3Mix_4s\dev-clean_4s\s3']  # List of source directories
    val_dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_val,
        source_dirs=source_dirs_val,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,  # Validation should not shuffle
            num_workers=config['num_workers'],
            pin_memory=True
        )

    # Initialize trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,  # ← ADDED: Pass the scheduler to the trainer
        device=config['device'],
        checkpoint_dir=config['checkpoint_dir'],
        log_dir=config['log_dir']
    )
    
    # Check for existing checkpoint if resuming
    resume_checkpoint = None
    if resume:
        resume_checkpoint = trainer.find_latest_checkpoint()
        if resume_checkpoint:
            print(f"Found checkpoint: {resume_checkpoint}")
        else:
            print("No checkpoint found, starting fresh")
    
    # Start training
    trainer.train(
        num_epochs=config['num_epochs'],
        resume_from=resume_checkpoint  # ← Pass checkpoint name
    )


if __name__ == "__main__":
    main(resume=False)


📁 Found 30458 matching audio files
📁 Found 3335 matching audio files
TRAINING CONV-TASNET
Device: cuda
Total Epochs: 100
Starting Epoch: 1
Training samples: 30458
Validation samples: 3335
Current LR: 0.001000
Best Val Loss: inf
Scheduler: ReduceLROnPlateau
  ✓ Checkpoint saved: checkpoint_epoch_0.pth

Epoch 1/100 (LR: 0.001000)
----------------------------------------------------------------------
  Batch 0/5077, Loss: 10.9928


KeyboardInterrupt: 

In [11]:
def main(resume: bool = False):
    config = {
        # Model parameters
        'num_sources': 3,
        'encoder_dim': 512,
        'feature_dim': 128,
        'sample_rate': 44100,
        'window_ms': 1,
        'num_layers': 8,
        'num_stacks': 4,
        'kernel_size': 3,
        'causal': False,
        'use_skip': True,
        
        # Training parameters
        'batch_size': 4,
        'num_epochs': 100,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        
        # System parameters
        'device': 'cuda',
        'num_workers': 0,
        'checkpoint_dir': './checkpoints',
        'log_dir': './logs',
    }

    # Initialize model
    model = ConvTasNet(
        num_sources=config['num_sources'],
        encoder_dim=config['encoder_dim'],
        feature_dim=config['feature_dim'],
        sample_rate=config['sample_rate'],
        window_ms=config['window_ms'],
        num_layers=config['num_layers'],
        num_stacks=config['num_stacks'],
        kernel_size=config['kernel_size'],
        causal=config['causal'],
        use_skip=config['use_skip']
    ).to(config['device'])
    
    # Initialize optimizer and scheduler
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=config['learning_rate'], 
        weight_decay=config['weight_decay']
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=3
    )
    
    # For training
    # Load mixed and source audio
    mixed_dir_train = "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\train_4s\mixture"
    source_dirs_train = ["D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\train_4s\drums", \
                        "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\train_4s\\bass", \
                        "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\train_4s\\vocals"]  # List of source directories
    train__dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_train,
        source_dirs=source_dirs_train,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    
    train_loader = DataLoader(
            train__dataset,
            batch_size=config['batch_size'],
            shuffle=True,
            num_workers=config['num_workers'],
            pin_memory=True
        )
    
    # For validation
    # Load mixed and source audio
    mixed_dir_val = "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\val_4s\mixture"
    source_dirs_val = ["D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\val_4s\drums", \
                       "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\val_4s\\bass", \
                       "D:\Datasets\musdb18hq\musdb18_rearranged\musdb18_4s\\val_4s\\vocals"]  # List of source directories
    val_dataset = PairedAudioDataset(
        mixed_dir=mixed_dir_val,
        source_dirs=source_dirs_val,
        sample_rate=config['sample_rate'],
        expected_duration=4.0
    )
    val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,  # Validation should not shuffle
            num_workers=config['num_workers'],
            pin_memory=True
        )

    # Initialize trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,  # ← ADDED: Pass the scheduler to the trainer
        device=config['device'],
        checkpoint_dir=config['checkpoint_dir'],
        log_dir=config['log_dir']
    )
    
    # Check for existing checkpoint if resuming
    resume_checkpoint = None
    if resume:
        resume_checkpoint = trainer.find_latest_checkpoint()
        if resume_checkpoint:
            print(f"Found checkpoint: {resume_checkpoint}")
        else:
            print("No checkpoint found, starting fresh")
    
    # Start training
    trainer.train(
        num_epochs=config['num_epochs'],
        resume_from=resume_checkpoint  # ← Pass checkpoint name
    )


if __name__ == "__main__":
    main(resume=False)


📁 Found 7272 matching audio files
📁 Found 900 matching audio files
TRAINING CONV-TASNET
Device: cuda
Total Epochs: 100
Starting Epoch: 1
Training samples: 7272
Validation samples: 900
Current LR: 0.001000
Best Val Loss: inf
Scheduler: ReduceLROnPlateau
  ✓ Checkpoint saved: checkpoint_epoch_0.pth

Epoch 1/100 (LR: 0.001000)
----------------------------------------------------------------------
  Batch 0/1818, Loss: 33.8344


KeyboardInterrupt: 

## Inference

In [17]:
config = {
    # Model parameters
    'num_sources': 2,
    'encoder_dim': 512,
    'feature_dim': 128,
    'sample_rate': 8000,
    'window_ms': 2,
    'num_layers': 8,
    'num_stacks': 3,
    'kernel_size': 3,
    'causal': False,
    'use_skip': True,
    
    # Training parameters
    'batch_size': 6,
    'num_epochs': 100,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    
    # System parameters
    'device': 'cuda',
    'num_workers': 4,
    'checkpoint_dir': './checkpoints',
    'log_dir': './logs',
}

model = ConvTasNet(
    num_sources=config['num_sources'],
    encoder_dim=config['encoder_dim'],
    feature_dim=config['feature_dim'],
    sample_rate=config['sample_rate'],
    window_ms=config['window_ms'],
    num_layers=config['num_layers'],
    num_stacks=config['num_stacks'],
    kernel_size=config['kernel_size'],
    causal=config['causal'],
    use_skip=config['use_skip']
)

In [18]:
checkpoint = torch.load('Trained Statistics/checkpoints/best_model.pth', map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [19]:
mixed_signal, _ = librosa.load('D:\Datasets\Source_Separation_Dataset\Data\LibriSpeech\Libri2Mix\dev-clean\mix_clean\84_174_241.wav', sr=config['sample_rate']) # evaluation
mixed_signal.shape

(63800,)

In [20]:
Audio(mixed_signal, rate=config['sample_rate'])

In [ ]:
model.eval()  # Set model to evaluation mode
with torch.no_grad():
    separated_signal = model(torch.FloatTensor(mixed_signal).unsqueeze(0))
separated_signal.shape

torch.Size([1, 2, 63800])

In [25]:
Audio(separated_signal[:,0,:].squeeze(), rate=config['sample_rate'])

In [ ]:
# =============================================================================
# SECTION 4: EVALUATION METRICS (SDR)
# =============================================================================

def calculate_sdr(
    estimated: torch.Tensor, 
    original: torch.Tensor,
    mask: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Calculate Signal-to-Distortion Ratio (SDR) for audio quality measurement.
    
    What is SDR?
    -----------
    SDR measures how well the estimated signal matches the original.
    Higher SDR = better separation quality.
    
    - SDR > 15 dB: Excellent separation
    - SDR 10-15 dB: Good separation  
    - SDR 5-10 dB: Fair separation
    - SDR < 5 dB: Poor separation
    
    How it works:
    -------------
    1. Find the optimal scaling to match estimated to original
    2. Calculate the "true signal" component (what matches)
    3. Calculate the "error" component (what doesn't match)
    4. SDR = 10 * log10(true_power / error_power)
    
    Parameters
    ----------
    estimated : torch.Tensor -- Estimated signal, shape (Batch, Samples)
    original : torch.Tensor -- Original signal, shape (Batch, Samples)
    mask : torch.Tensor, optional -- Binary mask to ignore certain samples, shape (Batch, Samples)
    
    Returns
    -------
    torch.Tensor -- SDR values in dB, shape (Batch,)
    """
    eps = 1e-8  # Small value to prevent division by zero
    
    # Apply mask if provided
    if mask is not None:
        original = original * mask
        estimated = estimated * mask
    
    # Calculate optimal scaling factor
    # This finds how much to scale the original to best match the estimated
    original_power = (original ** 2).sum(dim=1, keepdim=True) + eps
    scale = (original * estimated).sum(dim=1, keepdim=True) / original_power
    
    # Decompose into "true" and "error" components
    true_component = scale * original  # What matches
    error_component = estimated - true_component  # What doesn't match
    
    # Calculate powers
    true_power = (true_component ** 2).sum(dim=1)
    error_power = (error_component ** 2).sum(dim=1) + eps
    
    # SDR in decibels
    sdr = 10 * torch.log10(true_power / error_power)
    
    return sdr


def calculate_pit_sdr(
    estimated: torch.Tensor,
    original: torch.Tensor,
    mask: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Calculate SDR with Permutation Invariant Training (PIT).
    
    What is PIT?
    -----------
    When separating sources, we don't know which output corresponds to which 
    original source. For example, with 2 speakers:
    - Output 1 might be Speaker A or Speaker B
    - Output 2 might be Speaker B or Speaker A
    
    PIT tries all possible assignments and picks the best one!
    
    Parameters
    ----------
    estimated : torch.Tensor -- Estimated sources, shape (Batch, NumSources, Samples)
    original : torch.Tensor -- Original sources, shape (Batch, NumSources, Samples)
    mask : torch.Tensor, optional -- Binary mask, shape (Batch, Samples)
    
    Returns
    -------
    torch.Tensor
        Best average SDR for each batch item, shape (Batch,)
        
    Example
    -------
    >>> estimated = torch.randn(2, 2, 8000)  # 2 batches, 2 sources
    >>> original = torch.randn(2, 2, 8000)
    >>> sdr = calculate_pit_sdr(estimated, original)  # Shape: (2,)
    """
    batch_size, num_sources, num_samples = estimated.size()
    
    # Validate inputs
    assert original.size() == estimated.size(), "Shapes must match"
    assert num_sources < num_samples, "Axis 1 should be sources, axis 2 should be samples"
    
    # Zero-mean the signals (remove DC offset)
    estimated = estimated - estimated.mean(dim=2, keepdim=True)
    original = original - original.mean(dim=2, keepdim=True)
    
    # Generate all possible permutations
    # For 2 sources: [(0,1), (1,0)]
    # For 3 sources: [(0,1,2), (0,2,1), (1,0,2), (1,2,0), (2,0,1), (2,1,0)]
    all_permutations = list(permutations(range(num_sources)))
    
    # Calculate SDR for all source pairs
    # sdr_matrix[b, i, j] = SDR between estimated source i and original source j
    sdr_matrix = torch.zeros(batch_size, num_sources, num_sources, device=estimated.device)
    
    for i in range(num_sources):
        for j in range(num_sources):
            sdr_matrix[:, i, j] = calculate_sdr(estimated[:, i], original[:, j], mask)
    
    # Try all permutations and find the best total SDR
    sdr_per_permutation = []
    
    for perm in all_permutations:
        # Sum SDR for this permutation assignment
        # perm = (1, 0) means: estimated[0] ↔ original[1], estimated[1] ↔ original[0]
        sdr_sum = sum(sdr_matrix[:, idx, perm[idx]] for idx in range(num_sources))
        sdr_per_permutation.append(sdr_sum.unsqueeze(1))
    
    # Stack and find maximum
    all_sdr = torch.cat(sdr_per_permutation, dim=1)  # (Batch, NumPermutations)
    best_sdr, _ = all_sdr.max(dim=1)  # (Batch,)
    
    # Return average SDR per source
    return best_sdr / num_sources